# Antigravity Image Pipeline — Colab Worker

Run this notebook in Google Colab (with a GPU runtime) to act as a remote worker for your local frontend. It exposes the entire pipeline (Bicubic, Lanczos, and Real-ESRGAN) over a public URL.

In [ ]:
!pip install flask numpy pillow scipy matplotlib gradio opencv-python-headless
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install basicsr realesrgan

In [ ]:
import os
import time
import torch
import cv2
import numpy as np
from PIL import Image
import gradio as gr
from urllib.request import urlretrieve

# Ensure models directory exists
os.makedirs("models", exist_ok=True)

def ensure_realesrgan_model():
    model_path = "models/RealESRGAN_x4plus.pth"
    if not os.path.exists(model_path):
        print("Downloading RealESRGAN_x4plus.pth...")
        urlretrieve(
            "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
            model_path
        )
    return model_path

def enhance_image(image, method, scale, tile, face_enhance):
    """
    Run the pipeline on the uploaded image array (RGB, uint8).
    Returns the enhanced image array.
    """
    print(f"Processing with {method}, scale={scale}, tile={tile}")
    if method == "realesrgan":
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        
        model_path = ensure_realesrgan_model()
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        
        # Detect GPUs (up to 4)
        num_gpus = min(torch.cuda.device_count(), 4)
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device} (Available GPUs: {num_gpus})")

        upsampler = RealESRGANer(
            scale=4,
            model_path=model_path,
            model=model,
            tile=tile if tile > 0 else 0,
            tile_pad=10,
            pre_pad=0,
            half=torch.cuda.is_available(),
            device=device
        )
        
        bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        out_bgr, _ = upsampler.enhance(bgr, outscale=scale)
        return cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
    
    elif method in ["bicubic", "lanczos"]:
        # Simple fallback to PIL for CPU methods since they are fast anyway
        pil_img = Image.fromarray(image)
        resample = Image.BICUBIC if method == "bicubic" else Image.LANCZOS
        w, h = pil_img.size
        out = pil_img.resize((w * scale, h * scale), resample)
        return np.array(out)
    else:
        raise ValueError(f"Unknown method {method}")

# Create the Gradio interface
demo = gr.Interface(
    fn=enhance_image,
    inputs=[
        gr.Image(type="numpy", label="Input Image"),
        gr.Textbox(value="realesrgan", label="Method (bicubic, lanczos, realesrgan)"),
        gr.Number(value=4, label="Scale Factor"),
        gr.Number(value=0, label="Tile Size"),
        gr.Checkbox(value=False, label="Face Enhance (GFPGAN)")
    ],
    outputs=gr.Image(type="numpy", label="Enhanced Image"),
    title="Antigravity Colab Worker",
    description="API endpoint for offloading image processing from the local UI.",
    allow_flagging="never",
    api_name="enhance"
)

print("Starting Gradio with ngrok integration...")
demo.launch(share=True)
